Runnables have 3 types: 
1. Runnable Lambda
2. Runnable Passthrough
3. Runnable Parallel

### Runnable Lambda
Used when we want to add our own logic or function to be a part of the chain in LangChain. (Single Function only)

In [6]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableLambda

def make_Uppercase(text:str):
    return text.upper()

llm = init_chat_model(
    model="groq:qwen/qwen3.8-27b",
    max_tokens = 1024
)

prompt = ChatPromptTemplate.from_messages([
    {'role':'system','content':'You are a {topic} expert'},
    {'role':'user','content':'{query}'}
])

runnable = RunnableLambda(make_Uppercase)

chain = prompt | llm | StrOutputParser() | runnable

response = chain.invoke({"topic":"RAG","query":"Explain RAG in simple words"})

print(response)

IMAGINE YOU ARE A STUDENT TRYING TO PASS A VERY DIFFICULT EXAM, BUT YOU ARE **NOT ALLOWED TO MEMORIZE ANYTHING**. INSTEAD, YOU ARE ALLOWED TO TAKE ONE BOOK INTO THE EXAM HALL.

**RAG (RETRIEVAL-AUGMENTED GENERATION)** IS BASICALLY HOW AN AI USES THAT "BOOK."

HERE IS THE SIMPLE BREAKDOWN:

### 1. THE PROBLEM WITH JUST "MEMORIZING"
IF YOU JUST ASK AN AI A QUESTION WITHOUT RAG, IT RELIES ONLY ON WHAT IT "KNOWS" FROM ITS TRAINING DATA. THIS IS LIKE TRYING TO ANSWER A QUESTION FROM MEMORY.
*   **RISK:** IT MIGHT BE OUT OF DATE, MAKE THINGS UP (HALLUCINATE), OR NOT KNOW ABOUT YOUR SPECIFIC COMPANY’S INTERNAL DOCUMENTS.

### 2. HOW RAG FIXES THIS
RAG ADDS A STEP: **LOOK IT UP FIRST, THEN ANSWER.**

IT WORKS IN TWO MAIN STEPS:

*   **STEP 1: RETRIEVAL (THE "LOOK IT UP" PART)**
    WHEN YOU ASK A QUESTION, THE AI FIRST GOES INTO A SPECIFIC LIBRARY OF DOCUMENTS (LIKE YOUR COMPANY’S PDFS, HELP PAGES, OR DATABASES) AND FINDS THE MOST RELEVANT PAGES OR PARAGRAPHS.
    *   *EXAMPLE:* YOU ASK, "WHAT

### Runnable Passthrough
Passes the input without changing it. (Keeps the input intact)

In [10]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableLambda,RunnablePassthrough

def make_Uppercase(text:str):
    return text.upper()

llm = init_chat_model(
    model="groq:qwen/qwen3.8-27b",
    max_tokens = 500
)

prompt = ChatPromptTemplate.from_messages([
    {'role':'system','content':'You are a {topic} expert'},
    {'role':'user','content':'{query}'}
])

runnable = RunnableLambda(make_Uppercase)

run_pass = RunnablePassthrough()

# chain = prompt | llm | StrOutputParser() | run_pass | runnable

response = run_pass.invoke({"topic":"RAG","query":"Explain RAG in one sentence"})

print(response)

{'topic': 'RAG', 'query': 'Explain RAG in one sentence'}


### RunnableParallel
Used to run multiple function parallely

In [13]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableLambda,RunnablePassthrough,RunnableParallel

def make_Uppercase(text:str):
    return text.upper()

def count_length(text:str):
    return len(text)

llm = init_chat_model(
    model="groq:qwen/qwen3.8-27b",
    max_tokens = 500
)

prompt = ChatPromptTemplate.from_messages([
    {'role':'system','content':'You are a {topic} expert'},
    {'role':'user','content':'{query}'}
])

run_pass = RunnablePassthrough()

chain = prompt | llm | StrOutputParser() 

runnable_parallel = RunnableParallel(
    {
     "user_query":run_pass,
     "explaination": chain
    }

)

response = runnable_parallel.invoke({"topic":"RAG","query":"Explain RAG in simple words in one sentence"})

print(response)

{'user_query': {'topic': 'RAG', 'query': 'Explain RAG in simple words in one sentence'}, 'explaination': 'RAG is a technique that allows AI to answer questions by looking up relevant information from a specific database first, rather than relying solely on its pre-trained knowledge.'}


In [15]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableLambda,RunnablePassthrough,RunnableParallel

def make_Uppercase(text:str):
    return text.upper()

def count_length(text:str):
    return len(text)

llm = init_chat_model(
    model="groq:qwen/qwen3.8-27b",
    max_tokens = 500
)

prompt = ChatPromptTemplate.from_template("Explain {topic} in one sentence in simple words.")

run_pass = RunnablePassthrough()

chain = prompt | llm | StrOutputParser() 

runnable_parallel = RunnableParallel(
    {
     "topic":run_pass,
     "explaination": chain,
     "uppercase": chain | RunnableLambda(make_Uppercase),
     "count":chain | RunnableLambda(count_length)
    }

)

response = runnable_parallel.invoke({"topic":"RAG"})

print(response)

{'topic': {'topic': 'RAG'}, 'explaination': "RAG is like giving an AI a specific book to read first so it can answer your questions using only that book's information.", 'uppercase': 'RAG IS LIKE A STUDENT WHO IS ALLOWED TO OPEN THEIR TEXTBOOK TO FIND THE EXACT ANSWER DURING AN EXAM, INSTEAD OF HAVING TO RELY ONLY ON WHAT THEY MEMORIZED.', 'count': 179}


In the above code we have just printing the topic we passed in the input, but if we want the actual content/text of topic we have to use **.assign()**
assign takes the content out .

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.runnables import RunnableLambda,RunnablePassthrough,RunnableParallel

def make_Uppercase(text:str):
    return text.upper()

def count_length(text:str):
    return len(text)

llm = init_chat_model(
    model="groq:qwen/qwen3.8-27b",
    max_tokens = 500
)

prompt = ChatPromptTemplate.from_template("Explain {topic} in one sentence in simple words.")

chain = prompt | llm | StrOutputParser() 

# No need of RunnableParallel
run_pass = RunnablePassthrough.assign(
    explaination = chain,
    uppercase =  chain | RunnableLambda(make_Uppercase),
    count =  chain | RunnableLambda(count_length)
)

response = run_pass.invoke({"topic":"RAG"})

print(response)

{'topic': 'RAG', 'explaination': 'RAG is like a student who is allowed to open their textbook to look up specific answers before writing an exam, rather than relying only on what they already memorized.', 'uppercase': 'RAG HELPS AN AI ANSWER QUESTIONS ACCURATELY BY FIRST LOOKING UP RELEVANT INFORMATION IN A SPECIFIC DOCUMENT OR DATABASE AND THEN USING THAT INFORMATION TO CRAFT ITS RESPONSE.', 'count': 124}


<!--  -->